In [14]:
import logging
logging.getLogger("pdfminer").setLevel(logging.ERROR)

import os
from dotenv import load_dotenv

load_dotenv()

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "catcher-rag-eval"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

In [15]:
from pathlib import Path
import sys

def find_project_root() -> Path:
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr

PROJECT_ROOT = find_project_root()
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: c:\Users\user\catcher-llm


# 1. 문서 로드 — saving_tips 1~7.pdf

In [16]:
PDF_PATHS = [
    PROJECT_ROOT / "data/raw/pdf/saving_tips/1.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/2.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/3.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/4.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/5.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/6.pdf",
    PROJECT_ROOT / "data/raw/pdf/saving_tips/7.pdf",
]

In [17]:
from langchain_community.document_loaders import PDFPlumberLoader

all_docs = []
for path in PDF_PATHS:
    loader = PDFPlumberLoader(str(path))
    docs = loader.load()
    for d in docs:
        d.metadata["source"] = path.name
    all_docs.extend(docs)

print("총 문서 수:", len(all_docs))
print(all_docs[0].page_content[:300])

총 문서 수: 53
금융생활에 필요한 모든 정보,「파인」(fine.fss.or.kr)으로 검색하세요
“금융은 튼튼하게, 소비자는 행복하게”
보 도 참 고 자 료
보도 2017. 1. 31.(화) 조간 배포 2017. 1. 26.(목)
담당부서 금융혁신국 서정보 팀장(3145-8210) 이동춘 수석(3145-8216)
제 목 : 금융꿀팁 200선 -㉚ 사회초년생을 위한 금융꿀팁 7가지
□ 금융감독원은 국민들이 일상적인 금융거래과정에서 알아두면
유익한 실용금융정보(금융꿀팁) 200가지를 선정, 알기 쉽게 정리하여
◦ 매주 1~3가지씩 보도참고자료를 통해 


# 2. 문서 split

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=120
)
split_docs = text_splitter.split_documents(all_docs)
print(f"청킹 후 문서 수: {len(split_docs)}")

청킹 후 문서 수: 71


# 3. 임베딩 + 벡터 DB

In [19]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(split_docs, embeddings)
print(f"벡터 수: {vectorstore.index.ntotal}")

벡터 수: 71


# 4. Retriever 설정

In [20]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

# 5. 질문 → context → 답변 생성

In [21]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, max_tokens=300)

def run_rag(q):
    docs = retriever.invoke(q)
    context_texts = [doc.page_content[:400] for doc in docs]
    answer = llm.invoke(
        f"""질문: {q}

아래 문서에 있는 내용만 사용해서 핵심 답변을 1~2문장으로 작성하세요.
문서에 없는 내용은 절대 추가하지 마세요.

문서:
{context_texts}"""
    ).content
    sources = [doc.metadata.get("source", "출처 없음") for doc in docs]
    return answer, context_texts, sources

In [22]:
# 테스트
q = "사회초년생이 종잣돈을 모으려면 어떻게 해야 하나?"
answer, contexts, sources = run_rag(q)
print("답변:", answer)
print("출처:", sources)

답변: 사회초년생이 종잣돈을 모으려면, 월급에서 적은 금액이라도 꾸준히 저축하고, 자신의 소득과 투자성향을 고려하여 정기적금이나 적립식펀드에 가입하는 것이 중요합니다.
출처: ['1.pdf', '1.pdf', '1.pdf', '1.pdf']


In [23]:
def target(inputs: dict):
    q = inputs["question"]
    answer, contexts, sources = run_rag(q)
    return {"answer": answer, "contexts": contexts}

# 6. LangSmith Dataset 생성

In [24]:
from langsmith import Client

client = Client()
dataset_name = "catcher-rag-savingTips-eval"

questions = [
    "사회초년생이 종잣돈을 모으려면 어떻게 해야 하나?",
    "식비를 절약하는 가장 효과적인 방법은?",
    "불필요한 소비를 줄이는 실천 방법은?",
    "미니멀라이프를 시작하는 첫 단계는 무엇인가?",
    "할인 혜택을 잘 활용하는 방법은?",
]

# ⚠️ 아래 ground_truth는 문서를 읽고 실제 내용으로 수정하세요
ground_truths = [
    "꾸준히 저축하여 종잣돈을 모으는 것이 중요하다.",
    "계획적인 장보기와 식재료 관리로 식비를 절약할 수 있다.",
    "필요한 것과 원하는 것을 구분하여 소비를 줄여야 한다.",
    "물건을 필요한 만큼만 남기고 정리하는 것부터 시작한다.",
    "카드 할인, 포인트 적립 등 혜택을 미리 파악하고 활용한다.",
]

existing = [d for d in client.list_datasets() if d.name == dataset_name]
if existing:
    dataset = existing[0]
    print(f"기존 dataset 사용: {dataset.name}")
else:
    dataset = client.create_dataset(dataset_name=dataset_name, description="saving_tips 1~7 RAG 평가")
    for q, gt in zip(questions, ground_truths):
        client.create_example(
            inputs={"question": q},
            outputs={"ground_truth": gt},
            dataset_id=dataset.id
        )
    print(f"새 dataset 생성: {dataset.name} ({len(questions)}개)")

기존 dataset 사용: catcher-rag-savingTips-eval


# 7. Evaluator 정의

In [25]:
judge_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def correctness_evaluator(run, example):
    answer = run.outputs["answer"]
    ground_truth = example.outputs["ground_truth"]
    prompt = f"""다음 답변이 정답과 얼마나 일치하는지 0~1 점수로 평가해줘.
정답: {ground_truth}
답변: {answer}
숫자 하나만 출력해."""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "correctness", "score": float(score)}

def faithfulness_evaluator(run, example):
    answer = run.outputs["answer"]
    contexts = run.outputs["contexts"]
    prompt = f"""아래 답변이 문서에 있는 내용만 사용했는지 0~1로 평가해줘.
문서에 없는 정보를 추가했으면 낮게, 문서 내용만 사용했으면 높게.
숫자 하나만 출력해.
문서: {contexts}
답변: {answer}"""
    score = judge_llm.invoke(prompt).content.strip()
    return {"key": "faithfulness", "score": float(score)}

def actionability_evaluator(run, example):
    """구체적 실천 방법이 포함되어 있는가"""
    action_verbs = ["줄이기", "활용", "모으기", "정리", "계획", "구분", "점검", "해지", "대체", "고정"]
    answer = run.outputs["answer"]
    score = 1 if any(v in answer for v in action_verbs) else 0
    return {"key": "actionability", "score": score}

print("evaluator 3개 정의 완료")

evaluator 3개 정의 완료


# 8. evaluate() 실행 → LangSmith 반영

In [26]:
from langsmith.evaluation import evaluate

evaluate(
    target,
    data=dataset_name,
    evaluators=[correctness_evaluator, faithfulness_evaluator, actionability_evaluator],
    experiment_prefix="savingTips-rag-v1"
)

View the evaluation results for experiment: 'savingTips-rag-v1-ba148abf' at:
https://smith.langchain.com/o/f278f778-c6fd-420a-a653-490d4eb37548/datasets/35c859c8-62af-495d-953b-8c5e153e3dc0/compare?selectedSessions=963c3e36-a0ee-4bda-a34a-07e7b3f5ad2a




5it [00:27,  5.49s/it]


,inputs.question,outputs.answer,outputs.contexts,error,reference.ground_truth,feedback.correctness,feedback.faithfulness,feedback.actionability,execution_time,example_id,id
0,할인 혜택을 잘 활용하는 방법은?,할인 혜택을 잘 활용하려면 보험 가입 시 보험안내자료를 통해 할인제도를 꼼꼼히 확인...,[④ 자동이체 할인 특약\n- 보험료를 자동이체로 납입시 보험료 할인\n⑤ 수급권자...,None,"카드 할인, 포인트 적립 등 혜택을 미리 파악하고 활용한다.",0.6,1.0,1,5.325846,43e0352b-d68f-44a8-9d02-1a4a53acda0c,019ddd05-1f5f-7832-9388-94bdc770b4f0
1,미니멀라이프를 시작하는 첫 단계는 무엇인가?,미니멀라이프를 시작하는 첫 단계는 필요한 물건과 갖고 싶은 물건을 구분하여 불필요한...,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n미니멀라이프란 물건을 필...,None,물건을 필요한 만큼만 남기고 정리하는 것부터 시작한다.,0.6,1.0,1,2.721263,34a47569-c2f9-4607-8e42-b8b7a7cb6ad9,019ddd05-3d55-73d1-a286-de772eeb96ef
2,불필요한 소비를 줄이는 실천 방법은?,"불필요한 소비를 줄이기 위해 필요한 물건과 갖고 싶은 물건을 구분하고, 사용하지 않...",[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n미니멀라이프란 물건을 필...,None,필요한 것과 원하는 것을 구분하여 소비를 줄여야 한다.,0.8,1.0,1,3.890247,c45c29e4-d3f2-4f03-8db2-6eef771dda1f,019ddd05-50f5-7070-af33-3048a9493664
3,식비를 절약하는 가장 효과적인 방법은?,식비를 절약하는 가장 효과적인 방법은 계획적인 장보기를 통해 필요한 식재료를 미리 ...,[26. 4. 20. 오후 3:39 소비자시대 웹진\n\n# 절약생활의 시작은 식...,None,계획적인 장보기와 식재료 관리로 식비를 절약할 수 있다.,1.0,1.0,1,2.416100,0837a0da-1648-449e-8585-a93c45373d0c,019ddd05-657d-7af3-95b5-497666e5947e
4,사회초년생이 종잣돈을 모으려면 어떻게 해야 하나?,"사회초년생이 종잣돈을 모으려면, 월급에서 적은 금액이라도 꾸준히 저축하고, 자신의 ...",[제목 사회초년생을 위한 금융꿀팁 7가지\n⑤ 종잣돈 모으기\n사회생활을 시작하여 ...,None,꾸준히 저축하여 종잣돈을 모으는 것이 중요하다.,1.0,1.0,0,3.162727,58d2048d-806e-44b3-ae88-e45d4043d2e9,019ddd05-777b-7423-adee-4853d3ddea81
